# Tier 1: Computational & Physical Foundations
## Lab 1.1 — Biomolecular Databases, Sequence Literacy, and Quantitative Enzyme Kinetics

Welcome to **Lab 1.1** of the *Computational Structural Biology and Generative Macromolecular Design* curriculum!

### Why This Lab Matters
In Lab 1.0, we explored the physical layout of PDB coordinate records and electrostatic titration. Now, we expand from single coordinate files to the global ecosystem of biological databases and connect sequence information to quantitative enzymatic function.

Modern structural bioinformaticians must understand:
1. **Database Federation:** How primary observational archives (PDB, NCBI GenBank, TrEMBL) cross-reference curated secondary knowledge bases (UniProtKB/Swiss-Prot, AlphaFold DB, Rfam, RNAcentral).
2. **Four-Class Monomer Stereochemistry:** The building-block chemistry of life across amino acids, ribonucleotides, lipids (fatty acids), and monosaccharides.
3. **Sequence Homology vs. Sequence Identity:** How local alignment heuristics (BLAST) quantify evolutionary divergence via expect values ($E$-values), bit scores, and substitution matrices.
4. **Quantitative Function & Enzyme Modulation:** How sequence variations dictate Michaelis-Menten catalysis ($V_{\max}$, $K_m$), turnover rates ($k_{\mathrm{cat}}$), catalytic efficiency ($k_{\mathrm{cat}}/K_m$), and competitive vs. non-competitive inhibition modes.

---

### Learning Objectives
By the end of this laboratory, you will be able to:
- Programmatically query and parse REST APIs and XML streams across UniProt, PDB, and NCBI Entrez.
- Construct a four-class monomer lookup table mapping charge, H-bond donor/acceptor counts, and hydrophobicity.
- Calculate sliding-window Kyte-Doolittle hydrophobicity profiles for proteins and GC-content tracks for RNA.
- Run non-linear least squares (`scipy.optimize.curve_fit`) to extract Michaelis-Menten kinetic constants ($V_{\max}, K_m$) on PETase degradation datasets.
- Implement global non-linear fits across multi-$[I]$, multi-$[S]$ experimental curves to rigorously diagnose competitive vs. non-competitive enzyme inhibition modes.
- Evaluate your conceptual understanding using interactive `jupyterquiz` modules embedded directly into the notebook.

---

### Running Systems in Module 1
- **Protein Thread:** *Ideonella sakaiensis* **PETase** (PDB: `6ANE`, UniProt: `A0A0K8P6T7`), operating via its catalytic triad ($	ext{Ser160}$–$	ext{Asp206}$–$	ext{His237}$) and aromatic substrate clamp ($	ext{Trp185}$).
- **RNA Thread:** *Vibrio vulnificus* *add* **A-riboswitch** (PDB: `1Y26`, Rfam: `RF00167`, RNAcentral: `URS00003F6473`), evaluating adenine binding and discriminator pairing.

Let's begin by initializing our environment.


### Section 0: Environment Setup and Pinned Dependencies
We pin all core packages to guarantee exact numerical and algorithmic reproducibility across Google Colab runs.


In [ ]:
# Install required scientific bioinformatics packages (including valid jupyterquiz version)
!pip install -q biopython==1.83 jupyterquiz==2.8.1 py3Dmol==2.1.0 scipy numpy pandas matplotlib requests

import os
import sys
import json
import random
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from IPython.display import display

# Pinned reproducibility seed
COURSE_SEED = 42
random.seed(COURSE_SEED)
np.random.seed(COURSE_SEED)

# Entrez requires a contact email for live NCBI queries
from Bio import Entrez
Entrez.email = "student_training@university.edu"

print("Lab 1.1 environment initialized successfully.")
print(f"Reproducibility seed: {COURSE_SEED}")
print(f"Entrez registered email: {Entrez.email}")


---
## Section 1: The Four-Class Monomer Table and Physical Chemistry

All biomacromolecules are polymers synthesized from discrete monomeric subunits. Computational structural biology considers four fundamental classes:
1. **Amino Acids:** Peptidic monomers possessing an amino group, a carboxylate group, and one of 20 distinct side chains governing hydrophobicity, sterics, and charge.
2. **Nucleotides:** Ribonucleotide (RNA) or deoxyribonucleotide (DNA) monomers comprising a purine/pyrimidine base, a ribose sugar, and a phosphate group (polyanionic backbone).
3. **Lipids:** Amphipathic monomers (e.g., palmitate, glycerophospholipids) featuring a polar/charged headgroup and an extended aliphatic acyl tail.
4. **Polysaccharides (Monosaccharides):** Polyhydroxy aldehydes or ketones (e.g., D-glucose) with extensive hydrogen-bonding capacity and neutral charge at physiological pH.

Let's build a clean, queryable Python dictionary mapping monomer class, formal charge at pH $7.40$, hydrogen-bond donor/acceptor counts, and hydrophobicity flags.


In [ ]:
# Reference monomer property table across all four biological macromolecular classes
MONOMER_TABLE = {
    # 1. Amino Acids
    "alanine":   {"class": "amino_acid", "charge_pH74": 0, "hbond_donors": 1, "hbond_acceptors": 1, "hydrophobic": True},
    "aspartate": {"class": "amino_acid", "charge_pH74": -1, "hbond_donors": 1, "hbond_acceptors": 3, "hydrophobic": False},
    "lysine":    {"class": "amino_acid", "charge_pH74": +1, "hbond_donors": 3, "hbond_acceptors": 1, "hydrophobic": False},
    "histidine": {"class": "amino_acid", "charge_pH74": 0, "hbond_donors": 1, "hbond_acceptors": 1, "hydrophobic": False},
    # 2. Nucleotides (Free ribonucleotide 5'-monophosphate, AMP)
    # At pH 7.4, free terminal monophosphate carries ~ -2 charge (pKa2 ~ 6.2).
    # Inside a polymer chain, an internal phosphodiester linkage carries -1 charge.
    "amp":       {"class": "nucleotide", "charge_pH74": -2, "hbond_donors": 3, "hbond_acceptors": 7, "hydrophobic": False},
    # 3. Lipids (Fatty acid: charged carboxylate head + long hydrophobic acyl chain)
    "palmitate": {"class": "lipid",      "charge_pH74": -1, "hbond_donors": 1, "hbond_acceptors": 2, "hydrophobic": True},
    # 4. Monosaccharide (Polyhydroxy neutral sugar)
    "glucose":   {"class": "monosaccharide", "charge_pH74": 0, "hbond_donors": 5, "hbond_acceptors": 6, "hydrophobic": False}
}

def query_monomer(name: str) -> dict:
    """Returns structural and chemical metadata for a monomer."""
    key = name.strip().lower()
    if key not in MONOMER_TABLE:
        raise KeyError(f"Monomer '{name}' not found. Available: {list(MONOMER_TABLE.keys())}")
    return MONOMER_TABLE[key]

# Generate tabular DataFrame of four-class monomer primer
monomer_df = pd.DataFrame([{**{"Monomer": m.capitalize()}, **query_monomer(m)} for m in MONOMER_TABLE])
display(monomer_df)


### Formative Assessment: Monomer Properties across Biomolecule Classes
Test your comprehension of monomer ionization and polarity.


In [ ]:
from jupyterquiz import display_quiz

quiz_section_1 = [
    {
        "question": "Why does a free adenosine monophosphate (AMP) monomer carry a charge of approximately -2 at pH 7.4, whereas an internal adenosine nucleotide in an RNA strand carries a charge of -1?",
        "type": "multiple_choice",
        "answers": [
            {
                "answer": "A free nucleotide has a terminal monophosphate with two ionizable hydroxyls (pKa2 ~ 6.2), while an internal nucleotide forms two covalent phosphodiester bonds leaving only one ionizable oxygen.",
                "correct": True,
                "feedback": "Correct! Phosphodiester polymerization condenses one hydroxyl into a covalent ester bond, leaving exactly one negative charge per internal phosphate at neutral pH."
            },
            {
                "answer": "The adenine base is protonated in free solution but deprotonated when incorporated into an RNA strand.",
                "correct": False,
                "feedback": "Incorrect. The adenine base is neutral at pH 7.4 in both states (pKa of adenine N1 is ~3.5)."
            },
            {
                "answer": "Ribose sugars lose two protons upon polymerization into RNA.",
                "correct": False,
                "feedback": "Incorrect. The ribose 2'- and 3'-hydroxyl groups remain neutral (pKa > 12)."
            },
            {
                "answer": "Free AMP binds divalent magnesium, imparting an apparent -2 formal charge.",
                "correct": False,
                "feedback": "Incorrect. Mg2+ coordinates to neutralize negative charges; the -2 charge is an intrinsic property of the uncomplexed terminal phosphate."
            }
        ]
    },
    {
        "question": "Comparing a monosaccharide (glucose) with a saturated fatty acid (palmitate) at physiological pH 7.4, which statement is chemically accurate?",
        "type": "multiple_choice",
        "answers": [
            {
                "answer": "Glucose is neutral and highly hydrophilic due to 5 hydroxyl H-bond donors, whereas palmitate is an amphipathic anion carrying a -1 carboxylate head and a hydrophobic tail.",
                "correct": True,
                "feedback": "Correct! Glucose contains multiple polar hydroxyl groups with zero net charge, while fatty acids possess an ionized carboxylate head (-1) coupled to an aliphatic acyl chain."
            },
            {
                "answer": "Both molecules are non-polar hydrocarbons that partition exclusively into the core of lipid bilayers.",
                "correct": False,
                "feedback": "Incorrect. Glucose is water-soluble and does not spontaneously embed into hydrophobic membrane interiors."
            },
            {
                "answer": "Glucose carries a net charge of -5 at pH 7.4 due to deprotonation of all five hydroxyls.",
                "correct": False,
                "feedback": "Incorrect. Aliphatic hydroxyls do not deprotonate at neutral pH."
            },
            {
                "answer": "Palmitate is a zwitterion with no net electrical charge.",
                "correct": False,
                "feedback": "Incorrect. Palmitate has a carboxylate group but no basic amino group; it is an anion, not a zwitterion."
            }
        ]
    }
]

display_quiz(quiz_section_1)


---
## Section 2: Biomolecular Databases Tour in Code

Modern bioinformatics integrates multiple specialized repositories. Knowing where authoritative data resides is a crucial skill:

| Database | Primary Content | Category | Authoritative URL / Endpoint |
|---|---|---|---|
| **PDB (RCSB)** | Experimentally determined 3D atomic coordinates (X-ray, Cryo-EM, NMR) | Primary | `https://files.rcsb.org/download/` |
| **UniProtKB** | Comprehensive protein sequences, functional domains, active sites, mutations | Secondary / Curated | `https://rest.uniprot.org/uniprotkb/` |
| **AlphaFold DB** | AI-predicted protein 3D structures with per-residue pLDDT and pairwise PAE | Derived / Modeled | `https://alphafold.ebi.ac.uk/files/` |
| **Rfam** | Non-coding RNA families, consensus secondary structures, covariance models | Secondary / RNA | `https://rfam.org/family/` *(Note: `rfam.xfam.org` is retired)* |
| **RNAcentral** | Comprehensive non-coding RNA sequence identifier hub | Comprehensive Hub | `https://rnacentral.org/api/` |

### Programmatic Querying of UniProt via REST API
Let's retrieve the canonical entry for *Ideonella sakaiensis* PETase (UniProt ID: **`A0A0K8P6T7`**) in JSON format directly from the UniProt REST API.


In [ ]:
def fetch_uniprot_entry(accession: str) -> dict:
    """Fetches a curated protein record from the UniProt REST API with offline fallback."""
    url = f"https://rest.uniprot.org/uniprotkb/{accession}.json"
    headers = {"Accept": "application/json"}
    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        return response.json()
    except Exception as e:
        print(f"Notice: UniProt live REST query failed ({e}). Utilizing cached reference payload.")
        return {
            "proteinDescription": {"recommendedName": {"fullName": {"value": "Poly(ethylene terephthalate) hydrolase"}}},
            "organism": {"scientificName": "Ideonella sakaiensis (strain 201-F6)"},
            "sequence": {
                "value": "MNFPRASRLMQAAVLGGLMAVSAAATAQTNPYARGPNPTAASLEASAGPFTVRSFTVSRPSGYGAGTVYYPTNAGGTVGAIAIVPGYTARQSSIKWWGPRLASHGFVVITIDTNSTLDQPSSRSSQQMAALRQVASLNGTSSSPIYGKVDTARMGVMGWSMGGGGSLISAANNPSLKAAAPQAPWDSSTNFSSVTVPTLIFACENDSIAPVNSSALPIYDSMSRNAKQFLEINGGSHSCANSGNSNQALIGKKGVAWMKRFMDNDTRYSTFACENPNSTRVSDFRTANCS",
                "length": 290
            },
            "features": [
                {"type": "Signal", "location": {"start": {"value": 1}, "end": {"value": 27}}},
                {"type": "Active site", "location": {"start": {"value": 160}}, "description": "Proton acceptor / nucleophile"},
                {"type": "Active site", "location": {"start": {"value": 206}}, "description": "Charge relay system"},
                {"type": "Active site", "location": {"start": {"value": 237}}, "description": "Charge relay system"}
            ]
        }

# Fetch PETase record
petase_uniprot = fetch_uniprot_entry("A0A0K8P6T7")

# Extract functional annotation
protein_name = petase_uniprot["proteinDescription"]["recommendedName"]["fullName"]["value"]
organism = petase_uniprot["organism"]["scientificName"]
sequence = petase_uniprot["sequence"]["value"]
seq_len = petase_uniprot["sequence"]["length"]

print(f"UniProt ID: A0A0K8P6T7")
print(f"Protein Name: {protein_name}")
print(f"Organism: {organism}")
print(f"Sequence Length: {seq_len} amino acids")
print(f"Full Mature/Precursor Sequence:\n{sequence}")


### Resolving Numbering Offsets: Signal Peptides
Notice that the UniProt sequence length is **290 amino acids**, whereas our PDB `6ANE` crystal structure contains only **263 residues**.
Why?
- Residues 1–27 encode an N-terminal **signal peptide** that directs PETase secretion into the extracellular environment.
- In the mature secreted functional enzyme, the signal peptide is cleaved, starting at residue 28.
- The crystallographers numbered the first residue of their recombinant crystallized construct as residue 1 (corresponding to UniProt residue 28).
$$\text{Mature Sequence Index} = \text{Deposited PDB Index} + 27$$

Let's verify this mapping programmatically from the UniProt features list.


In [ ]:
# Scan features for signal peptide annotation
features = petase_uniprot.get("features", [])
signal_peptide = (1, 27)  # Verified default for PETase A0A0K8P6T7
active_sites = []

for feat in features:
    ftype = feat.get("type")
    if ftype == "Signal":
        signal_peptide = (feat["location"]["start"]["value"], feat["location"]["end"]["value"])
    elif ftype == "Active site":
        active_sites.append((feat["location"]["start"]["value"], feat.get("description", "")))

print(f"Annotated Signal Peptide: Residues {signal_peptide[0]} to {signal_peptide[1]} (cleaved upon secretion)")
print(f"Mature functional protein starts at UniProt residue {signal_peptide[1] + 1}")
print("\nAnnotated Catalytic Sites in UniProt:")
for pos, desc in active_sites:
    print(f"  UniProt #{pos} (Deposited PDB #{pos - 27}): {desc}")


### Formative Assessment: Database Selection & Numbering Contracts
Verify your knowledge of biological database organization.


In [ ]:
quiz_section_2 = [
    {
        "question": "You have sequenced a novel non-coding RNA and need to identify which RNA family and secondary-structure covariance model it belongs to. Which database should you consult?",
        "type": "multiple_choice",
        "answers": [
            {
                "answer": "Rfam (rfam.org), which classifies structured ncRNAs into families using statistical covariance models.",
                "correct": True,
                "feedback": "Correct! Rfam curates consensus secondary structures and covariance models for RNA families."
            },
            {
                "answer": "UniProtKB/Swiss-Prot.",
                "correct": False,
                "feedback": "Incorrect. UniProt is exclusively a protein sequence and functional annotation database."
            },
            {
                "answer": "AlphaFold DB.",
                "correct": False,
                "feedback": "Incorrect. AlphaFold DB hosts computationally modeled 3D protein structures, not RNA covariance models."
            },
            {
                "answer": "The PDB archive.",
                "correct": False,
                "feedback": "Incorrect. PDB archives specific experimentally determined 3D coordinate files, not generalized family covariance models."
            }
        ]
    },
    {
        "question": "What is the difference between a primary and a secondary biological database?",
        "type": "multiple_choice",
        "answers": [
            {
                "answer": "Primary databases archive raw experimental observations (e.g., PDB coordinates, GenBank reads); secondary databases add manual curation, computational predictions, and synthesized annotations (e.g., Swiss-Prot, Pfam).",
                "correct": True,
                "feedback": "Correct! Primary databases archive directly submitted experimental evidence, while secondary databases add curation, classification, and metadata."
            },
            {
                "answer": "Primary databases are open to the public; secondary databases require expensive proprietary paid subscriptions.",
                "correct": False,
                "feedback": "Incorrect. The major primary and secondary databases in molecular biology are openly accessible public goods."
            },
            {
                "answer": "Primary databases store DNA; secondary databases store only proteins.",
                "correct": False,
                "feedback": "Incorrect. Both primary and secondary databases exist for DNA, RNA, proteins, and small molecules."
            },
            {
                "answer": "Primary databases are error-free; secondary databases contain only speculative computationally hallucinated data.",
                "correct": False,
                "feedback": "Incorrect. Primary databases often archive experimental artifacts and errors; secondary curation resolves many of these errors."
            }
        ]
    }
]

display_quiz(quiz_section_2)


---
## Section 3: Sliding-Window Sequence Profiles (Hydrophobicity & GC Content)

Primary sequences dictate folding thermodynamics through local patterns of hydrophobicity and base pairing.

### Kyte-Doolittle Hydrophobicity Profile
The Kyte-Doolittle scale quantifies the free energy of transfer from water to an apolar solvent for each amino acid side chain:
- **Positive scores (up to $+4.5$ for Isoleucine):** Hydrophobic, core-burying, or transmembrane regions.
- **Negative scores (down to $-4.5$ for Arginine):** Highly polar or charged, solvent-exposed surfaces.

A sliding-window average smooths noisy single-residue values across a window size $W$ (typically $W = 9$ or $19$ residues):
$$H_i = \frac{1}{W} \sum_{j = i - \lfloor W/2 \rfloor}^{i + \lfloor W/2 \rfloor} s(a_j)$$

Let's compute the hydrophobicity profile of mature PETase.


In [ ]:
# Kyte-Doolittle hydrophobicity scale
KD_SCALE = {
    'A': 1.8, 'R': -4.5, 'N': -3.5, 'D': -3.5, 'C': 2.5,
    'Q': -3.5, 'E': -3.5, 'G': -0.4, 'H': -3.2, 'I': 4.5,
    'L': 3.8, 'K': -3.9, 'M': 1.9, 'F': 2.8, 'P': -1.6,
    'S': -0.8, 'T': -0.7, 'W': -0.9, 'Y': -1.3, 'V': 4.2
}

def compute_kd_profile(seq: str, window: int = 9) -> np.ndarray:
    """Calculates sliding-window Kyte-Doolittle hydrophobicity profile."""
    seq = seq.upper()
    half = window // 2
    profile = np.zeros(len(seq))
    for i in range(len(seq)):
        start = max(0, i - half)
        end = min(len(seq), i + half + 1)
        window_vals = [KD_SCALE[aa] for aa in seq[start:end] if aa in KD_SCALE]
        profile[i] = np.mean(window_vals) if window_vals else 0.0
    return profile

# Analyze mature PETase sequence (excluding signal peptide 1-27)
mature_petase_seq = sequence[27:]
kd_profile = compute_kd_profile(mature_petase_seq, window=9)

print(f"Mature PETase sequence length: {len(mature_petase_seq)} aa")
print(f"Hydrophobicity: Min = {kd_profile.min():.2f}, Max = {kd_profile.max():.2f}, Mean = {kd_profile.mean():.2f}")

plt.figure(figsize=(9, 3.5), dpi=100)
plt.plot(np.arange(1, len(kd_profile) + 1), kd_profile, color='#1f77b4', lw=1.8, label='Kyte-Doolittle (Window=9)')
plt.axhline(0, color='gray', linestyle='--', lw=1.0)
# Highlight active site residues
plt.scatter([160, 206, 237, 185], [kd_profile[159], kd_profile[205], kd_profile[236], kd_profile[184]], 
            color='red', zorder=5, label='Triad + Trp185')
plt.title("PETase Sliding-Window Hydrophobicity Profile", fontsize=12, fontweight='bold')
plt.xlabel("Mature Residue Number", fontsize=11)
plt.ylabel("Average Hydrophobicity", fontsize=11)
plt.grid(True, alpha=0.3)
plt.legend(frameon=True)
plt.tight_layout()
plt.show()


### RNA Base Composition & GC Content Track
For nucleic acids, local GC content dictates duplex thermal melting stability:
- **G-C pairs** form **3 hydrogen bonds** (higher thermodynamic stability).
- **A-U pairs** form **2 hydrogen bonds** (lower stability).

Let's compute the nucleotide composition and sliding-window GC content for the *add* A-riboswitch aptamer sequence.


In [ ]:
def compute_gc_track(rna_seq: str, window: int = 10) -> np.ndarray:
    """Calculates sliding-window GC fraction across an RNA sequence."""
    seq = rna_seq.upper().replace('T', 'U')
    half = window // 2
    gc_track = np.zeros(len(seq))
    for i in range(len(seq)):
        start = max(0, i - half)
        end = min(len(seq), i + half + 1)
        sub = seq[start:end]
        gc_count = sum(1 for nt in sub if nt in ('G', 'C'))
        gc_track[i] = gc_count / max(1, len(sub))
    return gc_track

# Purine riboswitch aptamer region sequence
riboswitch_seq = "GGACAUAUAAUCGCGUGGAUAUGGCACGCAAGUUUCUACCGGGCACCGUAAAUGUCCGACUAUGUCC"
gc_track = compute_gc_track(riboswitch_seq, window=10)

# Global base composition
base_counts = {b: riboswitch_seq.count(b) for b in "ACGU"}
global_gc = (base_counts['G'] + base_counts['C']) / len(riboswitch_seq)

print(f"Riboswitch Length: {len(riboswitch_seq)} nt")
print(f"Base Counts: {base_counts}")
print(f"Overall GC Content: {global_gc:.3f}")

plt.figure(figsize=(8, 3), dpi=100)
plt.plot(np.arange(1, len(gc_track) + 1), gc_track, color='#2ca02c', lw=2.0)
plt.axhline(global_gc, color='gray', linestyle=':', label=f'Mean GC ({global_gc:.2f})')
plt.title("Purine Riboswitch Sliding-Window GC-Content Track", fontsize=12, fontweight='bold')
plt.xlabel("Nucleotide Position", fontsize=11)
plt.ylabel("GC Fraction (Window=10)", fontsize=11)
plt.ylim(0, 1.0)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


---
## Section 4: Quantitative Enzyme Kinetics & Catalytic Specificity

Enzymes accelerate chemical reactions without altering equilibrium thermodynamics.

### The Michaelis-Menten Model
For a single-substrate catalyzed reaction:
$$\mathrm{E} + \mathrm{S} \underset{k_{-1}}{\overset{k_1}{\rightleftharpoons}} \mathrm{ES} \xrightarrow{k_{\mathrm{cat}}} \mathrm{E} + \mathrm{P}$$

Under the Briggs-Haldane steady-state assumption ($\frac{d[\mathrm{ES}]}{dt} \approx 0$):
$$v = \frac{V_{\max} [\mathrm{S}]}{K_m + [\mathrm{S}]}$$
Where:
- $V_{\max} = k_{\mathrm{cat}} [\mathrm{E}]_{\text{total}}$ is the maximum velocity at saturating substrate concentration.
- $K_m = \frac{k_{-1} + k_{\mathrm{cat}}}{k_1}$ is the Michaelis constant (the substrate concentration at which $v = \frac{1}{2} V_{\max}$).
- $k_{\mathrm{cat}} = \frac{V_{\max}}{[\mathrm{E}]_{\text{total}}}$ is the catalytic turnover number (units of $\text{time}^{-1}$).
- $k_{\mathrm{cat}} / K_m$ is the **specificity constant** (the second-order rate constant governing reaction efficiency at low $[\mathrm{S}]$).

### Fitting Experimental Kinetics with Non-Linear Least Squares
Historically, scientists linearized data using double-reciprocal **Lineweaver-Burk plots** ($\frac{1}{v}$ vs. $\frac{1}{[\mathrm{S}]}$). However, Lineweaver-Burk transformations distort experimental measurement errors, severely overweighting points at low $[\mathrm{S}]$. Modern biophysics requires **direct non-linear regression** using Levenberg-Marquardt algorithms (`scipy.optimize.curve_fit`).

Let's load an experimental kinetic dataset comparing PETase activity on two distinct substrates (Substrate 1: $p$-NP-butyrate, Substrate 2: bis(2-hydroxyethyl) terephthalate, BHET).


In [ ]:
# Experimental PETase steady-state kinetics data
# Total enzyme concentration [E]_total = 0.05 uM (0.00005 mM)
E_TOTAL_MM = 0.00005  # 50 nM

kinetics_data = {
    "substrate1_name": "p-NP-butyrate",
    "substrate1_conc_mM": np.array([0.05, 0.1, 0.2, 0.5, 1.0, 2.0, 5.0]),
    "substrate1_rate_uM_min": np.array([1.25, 2.15, 3.48, 5.24, 6.45, 7.30, 7.82]),
    
    "substrate2_name": "BHET",
    "substrate2_conc_mM": np.array([0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0]),
    "substrate2_rate_uM_min": np.array([0.48, 0.91, 1.85, 2.92, 4.10, 5.35, 5.88])
}

def michaelis_menten(S: np.ndarray, Vmax: float, Km: float) -> np.ndarray:
    """Michaelis-Menten rate law equation."""
    return (Vmax * S) / (Km + S)

def fit_kinetics(S_vals: np.ndarray, v_vals: np.ndarray) -> tuple:
    """Fits Vmax and Km using non-linear least squares."""
    # Initial parameter guesses
    p0 = [np.max(v_vals) * 1.1, np.median(S_vals)]
    bounds = (0, [np.inf, np.inf])
    popt, pcov = curve_fit(michaelis_menten, S_vals, v_vals, p0=p0, bounds=bounds)
    perr = np.sqrt(np.diag(pcov))
    return popt[0], popt[1], perr[0], perr[1]

# Fit Substrate 1
vmax1, km1, err_v1, err_k1 = fit_kinetics(kinetics_data["substrate1_conc_mM"], kinetics_data["substrate1_rate_uM_min"])
# Fit Substrate 2
vmax2, km2, err_v2, err_k2 = fit_kinetics(kinetics_data["substrate2_conc_mM"], kinetics_data["substrate2_rate_uM_min"])

# Calculate turnover number (kcat) and specificity constant (kcat / Km)
# Note: Vmax in uM/min = 0.001 mM/min
kcat1 = (vmax1 * 1e-3) / E_TOTAL_MM  # min^-1
kcat2 = (vmax2 * 1e-3) / E_TOTAL_MM

spec1 = kcat1 / km1  # mM^-1 min^-1
spec2 = kcat2 / km2

kinetics_summary = pd.DataFrame([
    {"Substrate": kinetics_data["substrate1_name"], "Vmax (uM/min)": f"{vmax1:.2f} ± {err_v1:.2f}", "Km (mM)": f"{km1:.2f} ± {err_k1:.2f}", "kcat (min^-1)": f"{kcat1:.1f}", "kcat/Km (mM^-1 min^-1)": f"{spec1:.1f}"},
    {"Substrate": kinetics_data["substrate2_name"], "Vmax (uM/min)": f"{vmax2:.2f} ± {err_v2:.2f}", "Km (mM)": f"{km2:.2f} ± {err_k2:.2f}", "kcat (min^-1)": f"{kcat2:.1f}", "kcat/Km (mM^-1 min^-1)": f"{spec2:.1f}"}
])

display(kinetics_summary)
print(f"Catalytic Preference (Substrate 1 / Substrate 2): {spec1 / spec2:.2f}-fold preference for {kinetics_data['substrate1_name']}")


### Saturation Curve Visualization
Let's inspect the saturation curves and verify the non-linear fits.


In [ ]:
# Plot non-linear Michaelis-Menten fits
plt.figure(figsize=(7, 4.2), dpi=100)

s_grid1 = np.linspace(0, 5.5, 200)
s_grid2 = np.linspace(0, 10.5, 200)

plt.scatter(kinetics_data["substrate1_conc_mM"], kinetics_data["substrate1_rate_uM_min"], color='#1f77b4', s=45, label=f"{kinetics_data['substrate1_name']} Data")
plt.plot(s_grid1, michaelis_menten(s_grid1, vmax1, km1), color='#1f77b4', lw=2.0, label='Fit (Substrate 1)')

plt.scatter(kinetics_data["substrate2_conc_mM"], kinetics_data["substrate2_rate_uM_min"], color='#ff7f0e', s=45, label=f"{kinetics_data['substrate2_name']} Data")
plt.plot(s_grid2, michaelis_menten(s_grid2, vmax2, km2), color='#ff7f0e', lw=2.0, label='Fit (Substrate 2)')

plt.title("Michaelis-Menten Substrate Saturation Kinetics", fontsize=12, fontweight='bold')
plt.xlabel("Substrate Concentration [S] (mM)", fontsize=11)
plt.ylabel("Initial Velocity v (μmol/min)", fontsize=11)
plt.grid(True, alpha=0.3)
plt.legend(frameon=True)
plt.tight_layout()
plt.show()


---
## Section 5: Enzyme Inhibition Modes (Competitive vs. Non-Competitive)

Inhibitors modulate enzyme kinetics through distinct thermodynamic mechanisms:

### 1. Competitive Inhibition
The inhibitor binds exclusively to the free enzyme ($\mathrm{E}$), competing directly with substrate at the active site:
$$v = \frac{V_{\max} [\mathrm{S}]}{K_m \left(1 + \frac{[\mathrm{I}]}{K_i}\right) + [\mathrm{S}]}$$
- **Diagnostic Signature:** $V_{\max}$ remains **unchanged** (saturating $[\mathrm{S}]$ overcomes the inhibitor). Apparent $K_m$ **increases** ($K_{m,\text{app}} = K_m(1 + [\mathrm{I}]/K_i)$).
- **Lineweaver-Burk:** Lines intersect at the **$y$-axis** ($1/V_{\max}$).

### 2. Pure Non-Competitive Inhibition
The inhibitor binds with identical affinity to both free enzyme ($\mathrm{E}$) and enzyme-substrate complex ($\mathrm{ES}$):
$$v = \frac{\frac{V_{\max}}{1 + [\mathrm{I}]/K_i} [\mathrm{S}]}{K_m + [\mathrm{S}]}$$
- **Diagnostic Signature:** $V_{\max}$ **decreases** ($V_{\max,\text{app}} = V_{\max} / (1 + [\mathrm{I}]/K_i)$). $K_m$ remains **unchanged**.
- **Lineweaver-Burk:** Lines intersect at the **$x$-axis** ($-1/K_m$).

### Global Multi-$[I]$, Multi-$[S]$ Model Selection
Let's fit both competitive and non-competitive models globally across a multi-inhibitor dataset and select the true mechanism using the sum of squared errors (SSE).


In [ ]:
# Multi-[I], Multi-[S] Inhibition Dataset
S_points = np.array([0.2, 0.5, 1.0, 2.0, 5.0])
inhibitor_concs = [0.0, 0.5, 2.0]  # mM

# Experimental reaction velocities v (uM/min)
velocity_matrix = np.array([
    [2.35, 4.41, 6.12, 7.55, 8.70],  # [I] = 0 mM
    [1.32, 2.80, 4.51, 6.20, 7.85],  # [I] = 0.5 mM
    [0.55, 1.28, 2.35, 3.88, 6.10]   # [I] = 2.0 mM
])

# Flatten data for global non-linear 2D curve fitting: (S, I) -> v
S_flat = np.tile(S_points, len(inhibitor_concs))
I_flat = np.repeat(inhibitor_concs, len(S_points))
v_flat = velocity_matrix.flatten()

def competitive_model(X, Vmax, Km, Ki):
    S, I = X
    return (Vmax * S) / (Km * (1.0 + I / Ki) + S)

def noncompetitive_model(X, Vmax, Km, Ki):
    S, I = X
    return ((Vmax / (1.0 + I / Ki)) * S) / (Km + S)

# Fit Competitive Model
p0 = [10.0, 0.5, 0.5]
popt_comp, _ = curve_fit(competitive_model, (S_flat, I_flat), v_flat, p0=p0, bounds=(0, np.inf))
v_pred_comp = competitive_model((S_flat, I_flat), *popt_comp)
sse_comp = np.sum((v_flat - v_pred_comp)**2)

# Fit Non-Competitive Model
popt_non, _ = curve_fit(noncompetitive_model, (S_flat, I_flat), v_flat, p0=p0, bounds=(0, np.inf))
v_pred_non = noncompetitive_model((S_flat, I_flat), *popt_non)
sse_non = np.sum((v_flat - v_pred_non)**2)

print(f"Competitive Model Fit:     SSE = {sse_comp:.4f} | Vmax = {popt_comp[0]:.2f}, Km = {popt_comp[1]:.2f} mM, Ki = {popt_comp[2]:.2f} mM")
print(f"Non-Competitive Model Fit: SSE = {sse_non:.4f} | Vmax = {popt_non[0]:.2f}, Km = {popt_non[1]:.2f} mM, Ki = {popt_non[2]:.2f} mM")

selected_mode = "Competitive" if sse_comp < sse_non else "Non-Competitive"
print(f"\nModel Selection Verdict: {selected_mode} inhibition is strongly supported by least-squares evidence.")


### Diagnostic Lineweaver-Burk Double-Reciprocal Plot
While non-linear regression provides rigorous parameter estimates, the Lineweaver-Burk plot serves as a diagnostic visual test for intersection patterns.


In [ ]:
plt.figure(figsize=(7, 4.2), dpi=100)
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

# Plot Lineweaver-Burk lines: 1/v vs 1/[S]
inv_S = 1.0 / S_points

for idx, conc in enumerate(inhibitor_concs):
    inv_v = 1.0 / velocity_matrix[idx]
    plt.scatter(inv_S, inv_v, color=colors[idx], s=40, label=f'[I] = {conc} mM')
    
    # Linear extrapolation
    slope, intercept = np.polyfit(inv_S, inv_v, 1)
    x_extrap = np.linspace(-1.5, 5.5, 100)
    plt.plot(x_extrap, slope * x_extrap + intercept, color=colors[idx], linestyle='--', lw=1.5)

plt.axhline(0, color='gray', lw=1.0)
plt.axvline(0, color='gray', lw=1.0)
plt.title("Lineweaver-Burk Diagnostic Plot (Competitive Intersection on Y-Axis)", fontsize=12, fontweight='bold')
plt.xlabel("1 / [S] (mM⁻¹)", fontsize=11)
plt.ylabel("1 / v (min / μmol)", fontsize=11)
plt.xlim(-1.5, 5.5)
plt.ylim(0, 2.0)
plt.grid(True, alpha=0.3)
plt.legend(frameon=True)
plt.tight_layout()
plt.show()


### Formative Assessment: Enzyme Kinetics & Modulation
Test your mastery of steady-state kinetics and inhibition mechanisms.


In [ ]:
quiz_section_3 = [
    {
        "question": "An enzyme inhibitor raises the apparent Km of the reaction from 0.4 mM to 1.8 mM but leaves the maximum velocity Vmax completely unchanged at saturating substrate. What is the inhibition mechanism?",
        "type": "multiple_choice",
        "answers": [
            {
                "answer": "Competitive inhibition.",
                "correct": True,
                "feedback": "Correct! Competitive inhibitors compete for the free active site; high substrate concentrations outcompete the inhibitor, reaching full Vmax."
            },
            {
                "answer": "Pure non-competitive inhibition.",
                "correct": False,
                "feedback": "Incorrect. Non-competitive inhibition lowers Vmax while leaving Km unchanged."
            },
            {
                "answer": "Uncompetitive inhibition.",
                "correct": False,
                "feedback": "Incorrect. Uncompetitive inhibition lowers both Vmax and Km in equal proportion."
            },
            {
                "answer": "Irreversible suicide inactivation.",
                "correct": False,
                "feedback": "Incorrect. Irreversible inactivation permanently removes active enzyme molecules, reducing Vmax."
            }
        ]
    },
    {
        "question": "Why is the catalytic efficiency constant kcat / Km considered the most biologically relevant parameter for evaluating enzyme specificity?",
        "type": "multiple_choice",
        "answers": [
            {
                "answer": "It represents the apparent second-order rate constant for the reaction between free enzyme and free substrate at low, physiological substrate concentrations far below saturation.",
                "correct": True,
                "feedback": "Correct! In vivo, [S] is often << Km, meaning reaction rates depend directly on kcat / Km."
            },
            {
                "answer": "It is independent of temperature and pH.",
                "correct": False,
                "feedback": "Incorrect. Both kcat and Km are strongly temperature- and pH-dependent."
            },
            {
                "answer": "It equals the total molecular weight of the enzyme complex.",
                "correct": False,
                "feedback": "Incorrect. kcat / Km is a kinetic rate constant, not a mass."
            },
            {
                "answer": "It guarantees that the enzyme operates via an induced-fit mechanism.",
                "correct": False,
                "feedback": "Incorrect. The specificity constant applies to both lock-and-key and induced-fit enzymes."
            }
        ]
    }
]

display_quiz(quiz_section_3)


---
## Section 6: Open-Ended Investigation & Lab Summary

### Open-Ended Challenge Questions (For Graded Laboratory Report)
1. **Uncompetitive vs. Mixed Inhibition Modeling:**  
   Extend the non-linear regression function in Section 5 to include **uncompetitive inhibition**:
   $$v = \frac{V_{\max} [\mathrm{S}]}{K_m + [\mathrm{S}]\left(1 + \frac{[\mathrm{I}]}{K_i'}\right)}$$
   Fit this third model to the experimental velocity dataset. Compare the resulting sum of squared errors (SSE) and explain why Lineweaver-Burk lines for an uncompetitive inhibitor run strictly parallel.
   
2. **Signal Peptide Cleavage & Electrostatics:**  
   Using the primary sequence of full precursor PETase (UniProt: `A0A0K8P6T7`, 290 residues) retrieved in Section 2, calculate the net charge of the 27-residue signal peptide at pH $7.40$. How does the hydrophobicity and charge of the signal peptide compare to the mature folded catalytic domain? Explain how this physicochemical profile facilitates translocation across bacterial membranes.

---

### Key Takeaways from Lab 1.1
- **Database Federation:** Biological data is distributed across primary experimental archives (PDB, GenBank) and secondary curated knowledgebases (UniProt, AlphaFold DB, Rfam).
- **Signal Peptides & Coordinate Offsets:** Literature numbering often differs from deposited PDB coordinate indices due to cleaved leader peptides or crystallization purification tags.
- **Physical Profiles:** Kyte-Doolittle hydrophobicity profiles locate solvent-exposed versus buried core regions, while RNA GC-content tracks reflect duplex thermal stability.
- **Non-Linear Kinetics:** Steady-state kinetics and inhibition mechanisms must be extracted via non-linear least squares regression rather than error-distorting double-reciprocal linearizations.

You are now prepared to advance to **Lab 1.2: Structure Visualization, Crystallographic Quality, and Structural Superposition**!


In [ ]:
# Laboratory execution lockfile manifest
import json

lab1_1_manifest = {
    "lab_id": "Lab_1.1_Databases_Sequence_Kinetics",
    "course_seed": COURSE_SEED,
    "python_version": sys.version,
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__,
    "petase_uniprot_accession": "A0A0K8P6T7",
    "mature_petase_length": len(mature_petase_seq),
    "fitted_vmax_substrate1": round(float(vmax1), 4),
    "fitted_km_substrate1": round(float(km1), 4),
    "fitted_kcat_substrate1": round(float(kcat1), 2),
    "fitted_vmax_substrate2": round(float(vmax2), 4),
    "fitted_km_substrate2": round(float(km2), 4),
    "fitted_kcat_substrate2": round(float(kcat2), 2),
    "selected_inhibition_mode": selected_mode,
    "competitive_ki_mM": round(float(popt_comp[2]), 4)
}

os.makedirs("manifests", exist_ok=True)
manifest_path = "manifests/lab1_1_lockfile.json"
with open(manifest_path, "w") as f:
    json.dump(lab1_1_manifest, f, indent=4)

print(f"Manifest written successfully to: {manifest_path}")
print(json.dumps(lab1_1_manifest, indent=2))
